In [42]:
import pyomo.environ as pyo
from pyomo.environ import SolverFactory
from pyomo.environ import *

In [43]:
cortadores = ['eletrico','gas']
requisitos = ['producao','montagem','embalagem']

maximo_horas = {
    requisitos[0]:10000*60,
    requisitos[1]: 15000*60,    
    requisitos[2]: 5000*60
}

minimo_entrega = {
    cortadores[0]: 30000,
    cortadores[1]: 15000
}

informacoes = {
    cortadores[0]:{
        requisitos[0]: 0.2*60,
        requisitos[1]: 0.3*60,
        requisitos[2]: 0.1*60,
    },
    cortadores[1]:{
        requisitos[0]: 0.4*60,
        requisitos[1]: 0.5*60,
        requisitos[2]: 0.1*60
    }
}

custo_fab = {
    cortadores[0]: 55,
    cortadores[1]: 85  
}

custo_compra = {
    cortadores[0]: 67,
    cortadores[1]: 95
}

In [50]:
model = pyo.ConcreteModel()

model.cortadores = pyo.Set(initialize=cortadores)
model.requisitos = pyo.Set(initialize=requisitos)
model.x = pyo.Var(model.cortadores, domain=pyo.NonNegativeIntegers)
model.y = pyo.Var(model.cortadores, domain=pyo.NonNegativeIntegers)

def obj(model):
    soma_custo_fab = sum(custo_fab[c] * model.x[c] for c in model.cortadores)
    soma_custo_compra = sum(custo_compra[c] * model.y[c] for c in model.cortadores)

    return soma_custo_fab + soma_custo_compra
model.obj = pyo.Objective(rule=obj, sense=pyo.minimize)

def restricao_requisitos(model, r):
    return sum(informacoes[c][r] * model.x[c] for c in model.cortadores)  <= maximo_horas[r]
model.restricao_requisitos = pyo.Constraint(model.requisitos, rule=restricao_requisitos)

def restricao_entrega(model, c):
    return model.x[c] + model.y[c] >= minimo_entrega[c]
model.restricao_entrega = pyo.Constraint(model.cortadores, rule=restricao_entrega)

In [51]:
model.pprint()

2 Set Declarations
    cortadores : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'eletrico', 'gas'}
    requisitos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'producao', 'montagem', 'embalagem'}

2 Var Declarations
    x : Size=2, Index=cortadores
        Key      : Lower : Value : Upper : Fixed : Stale : Domain
        eletrico :     0 :  None :  None : False :  True : NonNegativeIntegers
             gas :     0 :  None :  None : False :  True : NonNegativeIntegers
    y : Size=2, Index=cortadores
        Key      : Lower : Value : Upper : Fixed : Stale : Domain
        eletrico :     0 :  None :  None : False :  True : NonNegativeIntegers
             gas :     0 :  None :  None : False :  True : NonNegativeIntegers

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expr

In [52]:
opt = SolverFactory('cplex')
res=opt.solve(model,tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmpkhz_me3x.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmpvoe2f0il.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmpvoe2f0il.pyomo.lp
Objective sense      : Minimize
Variables            :       4  [General Integer: 4]
Objective nonzeros   :       4
Linear constraints   :       5  [Less: 3,  Greater: 2]
  Nonzeros           :      10
  RHS nonzeros       :       5

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 55

In [53]:
for m in model.cortadores:
    print(f'{m} - fabricar: {model.x[m].value}, comprar: {model.y[m].value}')

eletrico - fabricar: 30000.0, comprar: 0.0
gas - fabricar: 10000.0, comprar: 5000.0
